In [22]:
import requests,json,os,datetime,zipfile
import geopandas as gpd
import pandas as pd
from datetime import datetime, timedelta
from google.colab import files

## Download Data

In [ ]:
import pandas as pd
import urllib.parse

### Set up Base URL

In [ ]:
# Step 1: Base URL
base_url = ""

In [ ]:
# Step 2: Query for 2024 records (encode it safely)
where = "created_date >= '2024-01-01T00:00:00' AND created_date < '2025-01-01T00:00:00'"
encoded = urllib.parse.quote(where)

# Step 3: Combine into one full URL
full_url = f"{base_url}?$where={encoded}"

# Step 4: Load JSON data
data = pd.read_json(full_url)

# Step 5: Show and save
print(data.head())
data.to_csv("dallas_311_2024.csv", index=False)
print("✅ Saved as dallas_311_2024.csv")

## Inspect Data
## Number of Row

In [4]:
df = pd.read_csv("dallas_311_2024.csv")

In [ ]:
row_count = len(df)
print(f"Number of rows: {row_count}")

## Data Cleaning
### Modify the Column Names

In [7]:
## Keep want colname less 10 char
### {old name: new name}
column_mapping = {
    "service_request_number": "Ser_Num",
    "service_request_type": "Type",
    "created_date": "Date",
    "lat_location": "Lat_Long",
}

In [ ]:
def preprocess_data(df):
    """
    Preprocesses the input DataFrame containing 311 data.

    Parameters:
        df (DataFrame): Input DataFrame containing 311 service request data.

    Returns:
        DataFrame: Preprocessed DataFrame with separate latitude and longitude columns.
    """
    import pandas as pd

    # Convert 'created_date' column to datetime format
    df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')

    # Select desired columns
    want_col = ["service_request_number", "service_request_type", "created_date", "lat_location"]
    part_df = df[want_col].copy()

    # Extract latitude and longitude from 'lat_location' column
    # Example format: "(32.83108631962040000,-96.85465316346955000)"
    part_df[['latitude', 'longitude']] = (
        part_df['lat_location']
        .str.replace(r"[()]", "", regex=True)  # remove parentheses
        .str.split(",", expand=True)  # split into two columns
        .astype(float)  # convert strings to numeric
    )

    # Create a formatted date column
    part_df['date'] = part_df['created_date'].dt.strftime("%Y-%m-%d")

    return part_df

processed_df = preprocess_data(df)

# Rename the columns using the column_mapping dictionary
processed_df.rename(columns=column_mapping, inplace=True)
# processed_df.drop_duplicates(subset=['Ser_Num'],inplace=True)  ## Remove redundant records
processed_df.head()

### Practice in Class

In [20]:
### save the processed_df file as 'dallas_incidents_2024_updated.csv'
processed_df.to_csv("dallas_311_2024_updated.csv", index=False)

In [ ]:
# Create GeoDataFrame for the last 28 days
shp_incident = gpd.GeoDataFrame(processed_df, geometry=gpd.points_from_xy(processed_df.longitude,
                                                                    processed_df.latitude,  crs=''))


In [ ]:
import matplotlib.pyplot as plt

# Plot the shapefile points
fig, ax = plt.subplots(figsize=(8, 8))
shp_incident.plot(ax=ax, color="royalblue", markersize=5, alpha=0.6, edgecolor="none")

# Add title and labels
ax.set_title("Dallas 311 Service Requests", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Add grid


In [ ]:
shp_incident.to_file('dallas_311')

In [ ]:
!zip -r dallas_311.zip dallas_311/
from google.colab import files
files.download("dallas_311.zip")
